In [ ]:
!pip install opendatasets

In [ ]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/ronikdedhia/next-word-prediction")

Skipping, found downloaded files in "./next-word-prediction" (use force=True to force download)


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, Dropout

In [ ]:
tokenizer = Tokenizer()

In [ ]:
with open('/content/next-word-prediction/1661-0.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
# FIX 1: Define VOCAB_SIZE before using it
VOCAB_SIZE = 10000  # cap vocabulary at 10,000 most frequent words

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts([text])

# FIX 2: Compute actual vocab size dynamically from the fitted tokenizer
# word_index includes ALL words; we cap at VOCAB_SIZE (+1 for padding index 0)
total_words = min(len(tokenizer.word_index) + 1, VOCAB_SIZE)
print(f"Vocabulary size used: {total_words}")

Vocabulary size used: 8933


In [ ]:
input_sequences = []
for sentence in text.split('\n'):
  tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

  for i in range(1, len(tokenized_sentence)):
    input_sequences.append(tokenized_sentence[:i+1])

print(f"Total sequences: {len(input_sequences)}")

Total sequences: 101619


In [ ]:
max_len = max([len(x) for x in input_sequences])
print(f"Max sequence length: {max_len}")

Max sequence length: 20


In [ ]:
padded_input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')
print(f"Padded shape: {padded_input_sequences.shape}")

Padded shape: (101619, 20)


In [ ]:
X = padded_input_sequences[:,:-1]

In [ ]:
y = padded_input_sequences[:,-1]

In [ ]:
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (101619, 19)
y shape: (101619,)


In [ ]:
y = to_categorical(y,num_classes=total_words)
print(f"y one-hot shape: {y.shape}")

y one-hot shape: (101619, 8933)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM,Dense,Input,Dropout


In [ ]:
model = Sequential([
    Input(shape=(max_len - 1,)),
    Embedding(total_words, 100),
    LSTM(150, return_sequences=False),
    Dropout(0.2),
    Dense(total_words, activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 19, 100)        │       893,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 150)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8933)           │     1,348,883 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,392,783 (9.13 MB)

 Trainable params: 2,392,783 (9.13 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(X,y,epochs=100, batch_size=64,verbose=1)

Epoch 1/100
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.0638 - loss: 6.3794
Epoch 2/100
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.1097 - loss: 5.7398
Epoch 3/100
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.1349 - loss: 5.3869
Epoch 4/100
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - accuracy: 0.1526 - loss: 5.1151
Epoch 5/100
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.1671 - loss: 4.8781
Epoch 6/100
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - accuracy: 0.1791 - loss: 4.6586
Epoch 7/100
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - accuracy: 0.1916 - loss: 4.4624
Epoch 8/100
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.2049 - loss: 4.2693
Epoch 9/100
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.2185 - loss: 4.0894
Epoch 10/100
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.2336 - loss: 3.9164
Epoch 11/100
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.2523 - loss: 3.75

In [ ]:
def predict_next_n_words(seed_text, model, tokenizer, max_len, n=10):
    result = seed_text
    index_to_word = {idx: word for word, idx in tokenizer.word_index.items()}

    for _ in range(n):
        token_list = tokenizer.texts_to_sequences([result])[0]
        token_list = pad_sequences([token_list], maxlen=max_len - 1, padding='pre')
        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]
        predicted_word = index_to_word.get(predicted_index, '<OOV>')
        result += ' ' + predicted_word

    return result

# ── 10. Test ──────────────────────────────────────────────────────────────────
seed = "The adventure began when"
print(predict_next_n_words(seed, model, tokenizer, max_len, n=20))

The adventure began when i saw that she was ejected by the butler and now and the lady had left the whole thing too
